# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [8]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, recall_score, f1_score
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE


In [2]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?

In [5]:
# Define features (X) and target (y)
X = fraud.drop('fraud', axis=1)
y = fraud['fraud']

# Split data (stratify ensures the split maintains the original imbalance ratio)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [6]:
print("Target Distribution:")
print(y_train.value_counts())
print(f"\nPercentage of Fraud: {(y_train.sum() / len(y_train) * 100):.2f}%")

Target Distribution:
fraud
0.0    638818
1.0     61182
Name: count, dtype: int64

Percentage of Fraud: 8.74%


- **2.** Train a LogisticRegression.

In [ ]:
# 1. Define features and target
X = fraud.drop('fraud', axis=1)
y = fraud['fraud']

# 2. Split the data (stratify=y ensures the 8.74% ratio is kept in both train and test sets)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 3. Train the baseline Logistic Regression
# We increase max_iter to ensure convergence
log_reg_base = LogisticRegression(max_iter=1000, random_state=42)
log_reg_base.fit(X_train, y_train)

# 4. Make predictions
y_pred_base = log_reg_base.predict(X_test)

# 5. Evaluate
print("=== Baseline Model (Imbalanced Data) ===")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_base))

print("\nClassification Report:")
# We focus on the metrics for class '1' (Fraud)
print(classification_report(y_test, y_pred_base, digits=4))

=== Baseline Model (Imbalanced Data) ===

Confusion Matrix:
[[271937   1842]
 [ 10434  15787]]

Classification Report:
              precision    recall  f1-score   support

         0.0     0.9630    0.9933    0.9779    273779
         1.0     0.8955    0.6021    0.7200     26221

    accuracy                         0.9591    300000
   macro avg     0.9293    0.7977    0.8490    300000
weighted avg     0.9571    0.9591    0.9554    300000



**3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.

Why Accuracy is wrong here: With 91% legitimate transactions, a model that predicts "Legit" for every single row achieves 91% accuracy but catches 0% of the fraud. Our baseline had 95.9% accuracy, which hides the fact that it missed 40% of frauds.

The Correct Metrics:

Recall (Sensitivity): Crucial for fraud. "Of all the actual frauds, how many did we catch?"
Baseline Recall: 0.6021 (We missed ~40% of frauds).
F1-Score: The harmonic mean of Precision and Recall. It balances the need to catch fraud (Recall) with the need to not annoy customers by flagging legitimate transactions (Precision).
Baseline F1-Score: 0.7200.
Conclusion for Step 3: The model is suboptimal. While precision is high (0.8955), the Recall (0.6021) is too low for a financial security context. Missing 4,000+ fraud cases (see the confusion matrix: 10,434 false negatives) is a critical failure. This confirms we must proceed with balancing techniques (Steps 4, 5, and 6) to improve Recall.

- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 

In [9]:
# 1. Apply Random Oversampling to the TRAINING set only
# We create a new balanced X_train and y_train
ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

# Verify the balance (should be 1:1 ratio now)
print("Target distribution after Oversampling:")
print(y_train_ros.value_counts()) 
# Expected: ~638k for both 0 and 1

# 2. Train the model on this NEW balanced data
log_reg_ros = LogisticRegression(max_iter=1000, random_state=42)
log_reg_ros.fit(X_train_ros, y_train_ros)

# 3. Evaluate on the ORIGINAL test set (which is still imbalanced, representing reality)
y_pred_ros = log_reg_ros.predict(X_test)

print("\n=== Step 4: Random Oversampling Results ===")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_ros))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_ros, digits=4))

Target distribution after Oversampling:
fraud
0.0    638818
1.0    638818
Name: count, dtype: int64

=== Step 4: Random Oversampling Results ===
Confusion Matrix:
[[255519  18260]
 [  1357  24864]]

Classification Report:
              precision    recall  f1-score   support

         0.0     0.9947    0.9333    0.9630    273779
         1.0     0.5766    0.9482    0.7171     26221

    accuracy                         0.9346    300000
   macro avg     0.7856    0.9408    0.8401    300000
weighted avg     0.9582    0.9346    0.9415    300000



- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?

In [10]:
# 1. Apply Random Undersampling to the TRAINING set
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

# Verify the balance (should match the minority class count: ~61k)
print("Target distribution after Undersampling:")
print(y_train_rus.value_counts())

# 2. Train the model on this reduced dataset
log_reg_rus = LogisticRegression(max_iter=1000, random_state=42)
log_reg_rus.fit(X_train_rus, y_train_rus)

# 3. Evaluate on the ORIGINAL test set
y_pred_rus = log_reg_rus.predict(X_test)

print("\n=== Step 5: Random Undersampling Results ===")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rus))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rus, digits=4))

Target distribution after Undersampling:
fraud
0.0    61182
1.0    61182
Name: count, dtype: int64

=== Step 5: Random Undersampling Results ===
Confusion Matrix:
[[255527  18252]
 [  1327  24894]]

Classification Report:
              precision    recall  f1-score   support

         0.0     0.9948    0.9333    0.9631    273779
         1.0     0.5770    0.9494    0.7177     26221

    accuracy                         0.9347    300000
   macro avg     0.7859    0.9414    0.8404    300000
weighted avg     0.9583    0.9347    0.9417    300000



- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [11]:
# 1. Apply SMOTE to the TRAINING set
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Verify balance
print("Target distribution after SMOTE:")
print(y_train_smote.value_counts())

# 2. Train the model
log_reg_smote = LogisticRegression(max_iter=1000, random_state=42)
log_reg_smote.fit(X_train_smote, y_train_smote)

# 3. Evaluate
y_pred_smote = log_reg_smote.predict(X_test)

print("\n=== Step 6: SMOTE Results ===")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_smote))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_smote, digits=4))

Target distribution after SMOTE:
fraud
0.0    638818
1.0    638818
Name: count, dtype: int64

=== Step 6: SMOTE Results ===
Confusion Matrix:
[[255664  18115]
 [  1401  24820]]

Classification Report:
              precision    recall  f1-score   support

         0.0     0.9946    0.9338    0.9632    273779
         1.0     0.5781    0.9466    0.7178     26221

    accuracy                         0.9349    300000
   macro avg     0.7863    0.9402    0.8405    300000
weighted avg     0.9581    0.9349    0.9418    300000

